# Inspect dotplot export values

Use this notebook to manually inspect the `adata_spatial` object and compare values against the exported dotplot summary CSV.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [2]:
adata_path = "/home/nnataren/Documents/PhD/Bioinformatics/Banksy_py_fork/Banksy_py/data/xenium/processed/CK_skin_res/adata_spatial_CK_skin_res_0p5.h5ad"
export_csv = "data/xenium/processed/cross_sample_dotplot_exports/local_test_CK_skin_res_0p5_dotplot_summary.csv"
groupby = "labels_scaled_gaussian_pc20_nc0.20_r0.50"

gene = "VWF"
cluster = "0"

In [3]:
adata = ad.read_h5ad(adata_path)
export = pd.read_csv(export_csv)

adata

AnnData object with n_obs × n_vars = 6760 × 912
    obs: 'sample_name', 'x', 'y', 'cell_area', 'nucleus_area', 'transcript_counts', 'total_counts', 'nucleus_count', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'segmentation_method', 'nCount_Xenium', 'nFeature_Xenium', 'labels_scaled_gaussian_pc20_nc0.20_r0.50', 'cell type'
    var: 'gene_ids', 'feature_types', 'genome', 'is_nbr', 'k'
    obsm: 'reduced_pc_20', 'reduced_pc_20_umap', 'xy'

## Inspect object structure

In [4]:
print("adata shape:", adata.shape)
print("adata.raw exists:", adata.raw is not None)
print("groupby exists:", groupby in adata.obs.columns)
print("gene in adata.var_names:", gene in adata.var_names)

if adata.raw is not None:
    print("gene in adata.raw.var_names:", gene in adata.raw.var_names)

adata shape: (6760, 912)
adata.raw exists: False
groupby exists: True
gene in adata.var_names: True


In [5]:
adata.obs.head()

,sample_name,x,y,cell_area,nucleus_area,transcript_counts,total_counts,nucleus_count,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,segmentation_method,nCount_Xenium,nFeature_Xenium,labels_scaled_gaussian_pc20_nc0.20_r0.50,cell type
aaabjldc-1,CK_skin_res,134.886642,638.876343,205.099695,121.831567,765,766,1,0,0,0,1,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,765,90,0,0
aaaenhao-1,CK_skin_res,238.324539,723.201538,167.394225,108.239535,771,771,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,771,110,0,0
aaamkopm-1,CK_skin_res,190.484802,734.506287,78.300940,49.220314,287,287,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,287,67,0,0
aaamomnk-1,CK_skin_res,178.695374,720.238647,74.191721,52.336096,270,270,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,270,65,0,0
aabbmchc-1,CK_skin_res,237.545151,570.459290,41.453439,NaN,60,60,0,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,60,33,2,2


In [6]:
adata.var.head()

,gene_ids,feature_types,genome,is_nbr,k
ACER1,ENSG00000167769,Gene Expression,Unknown,False,-1
ACTA2,ENSG00000107796,Gene Expression,Unknown,False,-1
ADAM12,ENSG00000148848,Gene Expression,Unknown,False,-1
AHNAK2,ENSG00000185567,Gene Expression,Unknown,False,-1
AIF1,ENSG00000204472,Gene Expression,Unknown,False,-1


In [7]:
adata.obs[groupby].value_counts().sort_index()

labels_scaled_gaussian_pc20_nc0.20_r0.50
0    2172
1    1329
2    1109
3     887
4     668
5     595
Name: count, dtype: int64

## Pull out one gene and one cluster

In [8]:
def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)

if adata.raw is not None and gene in adata.raw.var_names:
    expr_source = adata.raw
    source_name = "raw"
else:
    expr_source = adata
    source_name = "X"

expr = to_dense(expr_source[:, [gene]].X).ravel()
clusters = adata.obs[groupby].astype(str)
mask = (clusters == str(cluster)).to_numpy()
cluster_expr = expr[mask]

print("expression source:", source_name)
print("gene:", gene)
print("cluster:", cluster)
print("number of cells in cluster:", mask.sum())
print("first 20 expression values:")
cluster_expr[:20]

expression source: X
gene: VWF
cluster: 0
number of cells in cluster: 2172
first 20 expression values:


array([ 0.07688852,  0.07406201, -0.3375445 , -0.3375445 , -0.3375445 ,
       -0.3375445 , -0.3375445 ,  0.24926978, -0.3375445 , -0.3375445 ,
       -0.3375445 , -0.3375445 , -0.3375445 ,  0.18313493,  0.40163091,
        0.70959121, -0.3375445 ,  2.02638102, -0.3375445 , -0.3375445 ])

In [9]:
manual_mean_expression = cluster_expr.mean()
manual_percent_expressing = (cluster_expr > 0).mean() * 100

manual_mean_expression, manual_percent_expressing

(-0.23399357946287724, 10.359116022099448)

## Compare with export CSV

In [10]:
export.head()

,sample,resolution,cluster_id,sample_cluster,groupby,gene,marker_group,mean_expression,percent_expressing,n_cells,expression_source,adata_path
0,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,VWF,Endothelial cells,-0.233994,10.359116,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...
1,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,CLDN5,Endothelial cells,-0.049528,0.414365,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...
2,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,CDH5,Endothelial cells,-0.231480,7.458564,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...
3,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,ACTA2,Pericyte,-0.388511,17.081031,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...
4,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,THY1,Pericyte,-0.384374,13.351750,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...


In [11]:
export_row = export[
    (export["gene"].astype(str) == gene)
    & (export["cluster_id"].astype(str) == str(cluster))
]

export_row

,sample,resolution,cluster_id,sample_cluster,groupby,gene,marker_group,mean_expression,percent_expressing,n_cells,expression_source,adata_path
0,CK_skin_res,0.5,0,CK_skin_res__r0.50__cluster_0,labels_scaled_gaussian_pc20_nc0.20_r0.50,VWF,Endothelial cells,-0.233994,10.359116,2172,X,/home/nnataren/Documents/PhD/Bioinformatics/Ba...


In [12]:
if not export_row.empty:
    print("manual mean:", manual_mean_expression)
    print("export mean:", export_row["mean_expression"].iloc[0])
    print("manual percent expressing:", manual_percent_expressing)
    print("export percent expressing:", export_row["percent_expressing"].iloc[0])
    print("manual n_cells:", mask.sum())
    print("export n_cells:", export_row["n_cells"].iloc[0])

manual mean: -0.23399357946287724
export mean: -0.2339935794628772
manual percent expressing: 10.359116022099448
export percent expressing: 10.359116022099448
manual n_cells: 2172
export n_cells: 2172


Layers with keys: 